In [8]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from ordered_set import OrderedSet

from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
    LinearGaussianTimeInvariantTransitionModel
)
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.types.detection import TrueDetection
from stonesoup.types.state import GaussianState
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.updater.kalman import KalmanUpdater
from stonesoup.hypothesiser.probability import PDAHypothesiser
from stonesoup.dataassociator.probability import JPDAwithLBP as JPDA
from stonesoup.initiator.simple import MultiMeasurementInitiator
from stonesoup.deleter.time import UpdateTimeDeleter
from stonesoup.tracker.simple import MultiTargetTracker

# ---- Augmented Measurement Model for AR(1) Noise ----
class Augmented2DMeasurement(LinearGaussian):
    def function(self, state_vector, noise=False, **kwargs):
        # Accept both State objects and numpy arrays
        if hasattr(state_vector, 'state_vector'):
            s = np.asarray(state_vector.state_vector).flatten()
        else:
            s = np.asarray(state_vector).flatten()
        vec = np.array([
            s[0] + s[4],
            s[2] + s[5]
        ]).reshape(2, 1)
        if noise is True:
            vec += self.rvs()
        return vec

    def jacobian(self, state_vector, **kwargs):
        H = np.zeros((2, 6))
        H[0, 0] = 1  # x
        H[0, 4] = 1  # n_x
        H[1, 2] = 1  # y
        H[1, 5] = 1  # n_y
        return H

# ---- KF Experiment Function ----
def run_kf_experiment(seed, num_steps, rho, process_noise_var, sigma):
    np.random.seed(seed)
    init_states = [
        [0, 1, 0, 1, 0, 0],
        [20, -1, 10, -1, 0, 0]
    ]
    ar1_model = LinearGaussianTimeInvariantTransitionModel(
        transition_matrix=np.array([[rho]]),
        covariance_matrix=np.array([[sigma**2]])
    )
    transition_model = CombinedLinearGaussianTransitionModel([
        ConstantVelocity(np.sqrt(process_noise_var)),
        ConstantVelocity(np.sqrt(process_noise_var)),
        ar1_model, ar1_model
    ])

    measurement_model = Augmented2DMeasurement(
        ndim_state=6,
        mapping=(0, 2),
        noise_covar=np.zeros((2, 2))
    )
    dt = 0.05
    timesteps = [datetime.now() + timedelta(seconds=k*dt) for k in range(num_steps + 1)]
    truths = OrderedSet()
    truth_paths = []
    for state in init_states:
        truth = GroundTruthPath([GroundTruthState(state, timestamp=timesteps[0])])
        for k in range(1, num_steps + 1):
            next_state = transition_model.function(truth[k-1], noise=True, time_interval=timedelta(seconds=dt))
            truth.append(GroundTruthState(next_state, timestamp=timesteps[k]))
        truths.add(truth)
        truth_paths.append(truth)
    all_measurements = []
    for k in range(num_steps + 1):
        measurement_set = set()
        for idx, truth in enumerate(truth_paths):
            s = truth[k].state_vector.flatten()
            meas_vec = np.array([s[0] + s[4], s[2] + s[5]])
            measurement_set.add(TrueDetection(
                state_vector=meas_vec.reshape(-1, 1),
                groundtruth_path=truth,
                timestamp=timesteps[k],
                measurement_model=measurement_model
            ))
        all_measurements.append(measurement_set)

    predictor = KalmanPredictor(transition_model)
    updater = KalmanUpdater(measurement_model)
    hypothesiser = PDAHypothesiser(
        predictor=predictor,
        updater=updater,
        clutter_spatial_density=1e-12,
        prob_detect=1.0,
        prob_gate=0.999
    )
    data_associator = JPDA(hypothesiser=hypothesiser)
    prior_cov = np.diag([100, 10, 100, 10, 25, 25])
    prior_state = GaussianState(np.array(init_states[0]).reshape(-1, 1), prior_cov, timestamp=timesteps[0])
    deleter = UpdateTimeDeleter(time_since_update=timedelta(seconds=10))
    initiator = MultiMeasurementInitiator(
        prior_state=prior_state,
        deleter=deleter,
        data_associator=data_associator,
        updater=updater,
        measurement_model=measurement_model,
        min_points=1
    )

    detector = iter(list(zip(timesteps, all_measurements)))
    tracker = MultiTargetTracker(
        initiator=initiator,
        deleter=deleter,
        data_associator=data_associator,
        updater=updater,
        detector=detector
    )

    tracks = set()
    for time, curr_tracks in tracker:
        tracks |= curr_tracks
    tracks = {track for track in tracks if len(track) > 3}
    tracks = list(tracks)

    truth_arrays = [np.array([[s.state_vector[0], s.state_vector[2]] for s in truth]) for truth in truth_paths]
    track_arrays = [np.array([[s.state_vector[0], s.state_vector[2]] for s in track]) for track in tracks]

    rmse_time = []
    for t in range(num_steps + 1):
        errors = []
        for truth in truth_arrays:
            min_err = None
            for track in track_arrays:
                if t < track.shape[0]:
                    err = np.linalg.norm(track[t] - truth[t])
                    if (min_err is None) or (err < min_err):
                        min_err = err
            if min_err is not None:
                errors.append(min_err ** 2)
        if errors:
            rmse_t = np.sqrt(np.mean(errors))
            rmse_time.append(rmse_t)
        else:
            rmse_time.append(np.nan)
    overall_rmse = np.nanmean(rmse_time)
    return np.array(rmse_time), overall_rmse

# --- Run 10 seeds and average ---
num_seeds = 10
seeds = np.random.randint(0, 1e9, num_seeds)
num_steps = 500
rho = 0.95
process_noise_var = 0.02
sigma = 3.0
all_rmse = []
overall_rmse_list = []
for seed in range(num_seeds):
    rmse_time, overall_rmse = run_kf_experiment(seed, num_steps, rho, process_noise_var, sigma)
    all_rmse.append(rmse_time)
    overall_rmse_list.append(overall_rmse)


all_rmse = np.vstack(all_rmse)
mean_rmse = np.nanmean(all_rmse, axis=0)

plt.figure(figsize=(8, 4))
plt.plot(mean_rmse, label="Mean RMSE (10 seeds)")
plt.xlabel("Time Step")
plt.ylabel("RMSE")
plt.title("Averaged RMSE vs Time over 10 seeds")
plt.grid(linestyle=":")
plt.legend()
plt.tight_layout()
plt.show()


print(f"Average of Mean Overall RMSEs: {np.mean(overall_rmse_list):.3f}")
print(mean_rmse)
# Define ranges for rho
"""
def get_rho_plot(seed, num_steps, rho, process_noise_var, sigma):
    rho_values = np.linspace(0.10, 0.999, 16)
    mean_overall_rmses = []
    for rho in rho_values:
        all_rmse = []
        overall_rmse_list = []
        np.random.seed(12345)
        seeds = np.random.randint(0, 1e9, num_seeds)
        for seed in seeds:
            rmse_time, overall_rmse = run_kf_experiment(seed, num_steps, rho, process_noise_var, sigma)
            all_rmse.append(rmse_time)
            overall_rmse_list.append(overall_rmse)
        mean_overall_rmses.append(np.mean(overall_rmse_list))

    plt.figure()
    plt.plot(rho_values, mean_overall_rmses, marker="o")
    plt.xlabel(r"AR(1) $\rho$")
    plt.ylabel("Mean Overall RMSE")
    plt.title("RMSE vs AR(1) Persistence ($\\rho$)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    return mean_overall_rmses

_ = get_rho_plot(seed, num_steps, rho, process_noise_var, sigma)
"""

/Users/priyankbehera/Desktop/ID_JPDAF/StoneSoup/stonesoup/dataassociator/probability.py:307: RuntimeWarning: divide by zero encountered in divide
  mu = likelihood_matrix[:, 1:] / s
/Users/priyankbehera/Desktop/ID_JPDAF/StoneSoup/stonesoup/dataassociator/probability.py:314: RuntimeWarning: invalid value encountered in subtract
  nu = 1 / (1 + np.sum(mu, axis=0, keepdims=True) - mu)
/Users/priyankbehera/Desktop/ID_JPDAF/StoneSoup/stonesoup/dataassociator/probability.py:317: RuntimeWarning: divide by zero encountered in log10
  d = np.max(np.abs(np.log10(nu / nu_tilde)))


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from ordered_set import OrderedSet

from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
    LinearGaussianTimeInvariantTransitionModel
)
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.types.detection import TrueDetection
from stonesoup.types.state import GaussianState
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.updater.kalman import KalmanUpdater
from stonesoup.hypothesiser.probability import PDAHypothesiser
from stonesoup.dataassociator.probability import JPDAwithLBP as JPDA
from stonesoup.initiator.simple import MultiMeasurementInitiator
from stonesoup.deleter.time import UpdateTimeDeleter
from stonesoup.tracker.simple import MultiTargetTracker

# ---- Augmented Measurement Model for AR(1) Noise ----
class Augmented2DMeasurement(LinearGaussian):
    def function(self, state_vector, noise=False, **kwargs):
        # Accept both State objects and numpy arrays
        if hasattr(state_vector, 'state_vector'):
            s = np.asarray(state_vector.state_vector).flatten()
        else:
            s = np.asarray(state_vector).flatten()
        vec = np.array([
            s[0] + s[4],
            s[2] + s[5]
        ]).reshape(2, 1)
        if noise is True:
            vec += self.rvs()
        return vec

    def jacobian(self, state_vector, **kwargs):
        H = np.zeros((2, 6))
        H[0, 0] = 1  # x
        H[0, 4] = 1  # n_x
        H[1, 2] = 1  # y
        H[1, 5] = 1  # n_y
        return H

def run_kf_experiment_mismatch(seed, num_steps,
                               rho_true, sigma_true,      # Used for data generation
                               rho_filter, sigma_filter,  # Used for filtering
                               process_noise_var):
    np.random.seed(seed)
    init_states = [
        [0, 1, 0, 1, 0, 0],
        [20, -1, 10, -1, 0, 0]
    ]
    # 1. Truth uses "true" parameters
    ar1_model_truth = LinearGaussianTimeInvariantTransitionModel(
        transition_matrix=np.array([[rho_true]]),
        covariance_matrix=np.array([[sigma_true**2]])
    )
    transition_model_truth = CombinedLinearGaussianTransitionModel([
        ConstantVelocity(np.sqrt(process_noise_var)),
        ConstantVelocity(np.sqrt(process_noise_var)),
        ar1_model_truth, ar1_model_truth
    ])
    # 2. Filter uses "mismatched" parameters
    ar1_model_filter = LinearGaussianTimeInvariantTransitionModel(
        transition_matrix=np.array([[rho_filter]]),
        covariance_matrix=np.array([[sigma_filter**2]])
    )
    transition_model_filter = CombinedLinearGaussianTransitionModel([
        ConstantVelocity(np.sqrt(process_noise_var)),
        ConstantVelocity(np.sqrt(process_noise_var)),
        ar1_model_filter, ar1_model_filter
    ])

    measurement_model = Augmented2DMeasurement(
        ndim_state=6,
        mapping=(0, 2),
        noise_covar=np.zeros((2, 2))
    )
    dt = 0.05
    timesteps = [datetime.now() + timedelta(seconds=k*dt) for k in range(num_steps + 1)]
    # --- Generate truth and measurements with TRUE model ---
    truths = OrderedSet()
    truth_paths = []
    for state in init_states:
        truth = GroundTruthPath([GroundTruthState(state, timestamp=timesteps[0])])
        for k in range(1, num_steps + 1):
            next_state = transition_model_truth.function(truth[k-1], noise=True, time_interval=timedelta(seconds=0.05))
            truth.append(GroundTruthState(next_state, timestamp=timesteps[k]))
        truths.add(truth)
        truth_paths.append(truth)
    all_measurements = []
    for k in range(num_steps + 1):
        measurement_set = set()
        for idx, truth in enumerate(truth_paths):
            s = truth[k].state_vector.flatten()
            meas_vec = np.array([s[0] + s[4], s[2] + s[5]])
            measurement_set.add(TrueDetection(
                state_vector=meas_vec.reshape(-1, 1),
                groundtruth_path=truth,
                timestamp=timesteps[k],
                measurement_model=measurement_model
            ))
        all_measurements.append(measurement_set)

    # --- Filtering using MISMATCHED model ---
    predictor = KalmanPredictor(transition_model_filter)
    updater = KalmanUpdater(measurement_model)
    hypothesiser = PDAHypothesiser(
        predictor=predictor,
        updater=updater,
        clutter_spatial_density=1e-12,
        prob_detect=1.0,
        prob_gate=0.999
    )
    data_associator = JPDA(hypothesiser=hypothesiser)
    prior_cov = np.diag([100, 10, 100, 10, 25, 25])
    prior_state = GaussianState(np.array(init_states[0]).reshape(-1, 1), prior_cov, timestamp=timesteps[0])
    deleter = UpdateTimeDeleter(time_since_update=timedelta(seconds=10))
    initiator = MultiMeasurementInitiator(
        prior_state=prior_state,
        deleter=deleter,
        data_associator=data_associator,
        updater=updater,
        measurement_model=measurement_model,
        min_points=1
    )

    detector = iter(list(zip(timesteps, all_measurements)))
    tracker = MultiTargetTracker(
        initiator=initiator,
        deleter=deleter,
        data_associator=data_associator,
        updater=updater,
        detector=detector
    )

    tracks = set()
    for time, curr_tracks in tracker:
        tracks |= curr_tracks
    tracks = {track for track in tracks if len(track) > 3}
    tracks = list(tracks)

    truth_arrays = [np.array([[s.state_vector[0], s.state_vector[2]] for s in truth]) for truth in truth_paths]
    track_arrays = [np.array([[s.state_vector[0], s.state_vector[2]] for s in track]) for track in tracks]

    rmse_time = []
    for t in range(num_steps + 1):
        errors = []
        for truth in truth_arrays:
            min_err = None
            for track in track_arrays:
                if t < track.shape[0]:
                    err = np.linalg.norm(track[t] - truth[t])
                    if (min_err is None) or (err < min_err):
                        min_err = err
            if min_err is not None:
                errors.append(min_err ** 2)
        if errors:
            rmse_t = np.sqrt(np.mean(errors))
            rmse_time.append(rmse_t)
        else:
            rmse_time.append(np.nan)
    overall_rmse = np.nanmean(rmse_time)
    return np.array(rmse_time), overall_rmse

num_seeds = 10
num_steps = 500

# "Truth" parameters (used in data generation)
rho_true = 0.95
sigma_true = 1.5

# "Filter" parameters (used in the filter model -- mismatch here!)
rho_filter = 0.90
sigma_filter = 0.5

process_noise_var = 0.02

all_rmse = []
overall_rmse_list = []
for seed in range(num_seeds):
    rmse_time, overall_rmse = run_kf_experiment_mismatch(
        seed, num_steps, rho_true, sigma_true, rho_filter, sigma_filter, process_noise_var)
    all_rmse.append(rmse_time)
    overall_rmse_list.append(overall_rmse)

all_rmse = np.vstack(all_rmse)
mean_rmse = np.nanmean(all_rmse, axis=0)

"""
plt.figure(figsize=(8, 4))
plt.plot(mean_rmse, label="Mean RMSE (mismatched params)")
plt.xlabel("Time Step")
plt.ylabel("RMSE")
plt.title("Averaged RMSE with Parameter Mismatch")
plt.grid(linestyle=":")
plt.legend()
plt.tight_layout()
plt.show()
"""

print(f"Average of Mean Overall RMSEs: {np.mean(overall_rmse_list):.3f}")
print(mean_rmse)


Average of Mean Overall RMSEs: 8.470
[ 0.          0.12976595  0.2351503   0.39379682  0.47033761  0.55911566
  0.55528101  0.61954529  0.65587398  0.62758797  0.7018878   0.80333466
  0.81720411  0.85079034  0.86372906  0.9090393   0.92624185  0.89968784
  0.91288023  0.87211075  0.85513637  0.87552982  0.84267648  0.76169442
  0.7341309   0.74610421  0.77489487  0.808433    0.88879502  0.99687504
  1.06375411  1.12254873  1.16278329  1.13759682  1.10355209  1.09662752
  1.05797218  1.06317673  1.11323119  1.19922669  1.18434018  1.13717128
  1.08535189  1.07156812  1.0894619   1.11322913  1.13401088  1.20011602
  1.21446226  1.25644115  1.26169545  1.25270792  1.22451682  1.19923061
  1.20133861  1.20223851  1.16881124  1.13974273  1.12979082  1.14819831
  1.09707115  1.11838554  1.17788828  1.24085909  1.26861901  1.29678187
  1.31631285  1.33872818  1.35432892  1.36843104  1.51219587  1.52289428
  1.51249809  1.5016008   1.52158381  1.59758484  1.62932851  1.68701665
  1.69276982  

/var/folders/wk/r_3dm28j0wx2y0rl5hyq6bzc0000gn/T/ipykernel_25743/124830264.py:191: RuntimeWarning: Mean of empty slice
  mean_rmse = np.nanmean(all_rmse, axis=0)
